# 02 - Time-Series Analysis and Validation

## 0. Imports and upload

In [ ]:
                                                     
                                          

from google.colab import files
uploaded = files.upload()
input_file = next(iter(uploaded))

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew, kurtosis, pearsonr, spearmanr
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import TimeSeriesSplit, GroupKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import SelectKBest, mutual_info_regression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

OUT_DIR = "pm25_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

df_raw = pd.read_csv(input_file)
print("Original shape:", df_raw.shape)
print("Original columns:", df_raw.columns.tolist())
display(df_raw.head())

## 1. Standardize fields and validate the dataset

In [ ]:
                                                                                            
ALIASES = {
    "PM2.5": "pm25", "PM25": "pm25", "PM_25": "pm25",
    "station_id": "id", "station": "id", "station_name": "sensor_name",
    "lat": "latitude", "lon": "longitude", "lng": "longitude",
    "temperature": "temperature_celsius", "temp": "temperature_celsius",
    "pressure": "pressure_mb", "wind_u_component": "wind_u", "wind_v_component": "wind_v",
    "no2": "NO2", "co": "CO", "o3": "O3"
}
df = df_raw.rename(columns={c: ALIASES[c] for c in df_raw.columns if c in ALIASES}).copy()

required_core = ["date", "id", "pm25"]
missing_core = [c for c in required_core if c not in df.columns]
if missing_core:
    raise ValueError(f"Missing required columns: {missing_core}. Check the CSV headers and ALIASES.")

df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
if "datetime_utc" in df.columns:
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], errors="coerce")

numeric_candidates = [
    "latitude", "longitude", "temperature_celsius", "pressure_mb",
    "wind_u", "wind_v", "NO2", "CO", "O3", "AOD", "pm25"
]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

                                                                                                
df["id"] = df["id"].astype("string")
if "sensor_name" in df.columns:
    df["sensor_name"] = df["sensor_name"].astype("string")

print("Date range before PM2.5 filtering:", df["date"].min(), "to", df["date"].max())
print("Rows before cleaning:", len(df))
print("Stations before cleaning:", df["id"].nunique(dropna=True))
display(df.dtypes.to_frame("dtype"))

## 2. Mandatory PM2.5 cleaning - before EDA and splitting

In [ ]:
n_initial = len(df)
missing_pm25 = int(df["pm25"].isna().sum())
invalid_low = int((df["pm25"].notna() & (df["pm25"] <= 0)).sum())
invalid_high = int((df["pm25"].notna() & (df["pm25"] >= 500)).sum())
invalid_date = int(df["date"].isna().sum())
missing_station = int(df["id"].isna().sum())

                                                          
df = df[df["pm25"].notna() & df["pm25"].gt(0) & df["pm25"].lt(500)].copy()
df = df[df["date"].notna() & df["id"].notna()].copy()
df = df.sort_values(["id", "date"]).reset_index(drop=True)

cleaning_counts = pd.DataFrame({
    "reason": ["Missing PM2.5", "PM2.5 <= 0", "PM2.5 >= 500",
               "Invalid/missing date", "Missing station ID", "Rows retained"],
    "count": [missing_pm25, invalid_low, invalid_high, invalid_date,
              missing_station, len(df)]
})
display(cleaning_counts)
cleaning_counts.to_csv(f"{OUT_DIR}/PM25_cleaning_counts.csv", index=False)

print("Cleaned shape:", df.shape)
print("Cleaned date coverage:", df.date.min(), "to", df.date.max())
print("Stations:", df.id.nunique())
assert df["pm25"].gt(0).all() and df["pm25"].lt(500).all()

## 13. Temporal diagnostics - regularity, missing dates, ACF/PACF, Ljung-Box and ADF

In [ ]:
temporal_rows = []
for sid, g in df.groupby("id"):
    g = g.sort_values("date").drop_duplicates("date", keep="first")
    s = g.set_index("date")["pm25"].asfreq("D")
    observed = s.dropna()
    if observed.empty:
        continue
    intervals = observed.index.to_series().diff().dt.total_seconds().dropna()
    full_days = len(s)
    missing_days = int(s.isna().sum())

                                                                    
    mask = s.notna().to_numpy()
    best_start = best_len = run_start = run_len = 0
    for i, ok in enumerate(mask):
        if ok:
            if run_len == 0: run_start = i
            run_len += 1
            if run_len > best_len: best_start, best_len = run_start, run_len
        else:
            run_len = 0
    segment = s.iloc[best_start:best_start + best_len].dropna().to_numpy(float)

    acf1 = acf7 = pacf1 = lb_p = adf_p = np.nan
    if len(observed) >= 3:
        acf1 = s.autocorr(1)
    if len(observed) >= 8:
        acf7 = s.autocorr(7)
    if len(segment) >= 30 and np.nanstd(segment) > 0:
        try: pacf1 = float(pacf(segment, nlags=1, method="ywm")[1])
        except Exception: pass
        try:
            lb_lag = min(7, max(1, len(segment)//5))
            lb_p = float(acorr_ljungbox(segment, lags=[lb_lag], return_df=True)["lb_pvalue"].iloc[0])
        except Exception: pass
        try: adf_p = float(adfuller(segment, autolag="AIC")[1])
        except Exception: pass

    temporal_rows.append({
        "id": str(sid), "n_observed": len(observed),
        "first_date": observed.index.min(), "last_date": observed.index.max(),
        "expected_daily_dates": full_days, "missing_calendar_days": missing_days,
        "missing_calendar_days_pct": missing_days/full_days*100 if full_days else np.nan,
        "median_observed_interval_days": intervals.median()/86400 if len(intervals) else np.nan,
        "duplicate_timestamps_before_station_dedup": int(g.duplicated("date").sum()),
        "longest_uninterrupted_segment_days": len(segment),
        "acf_lag1": acf1, "acf_lag7": acf7, "pacf_lag1": pacf1,
        "ljung_box_p": lb_p, "adf_p": adf_p,
        "pct_above_25": (observed > 25).mean()*100,
        "pct_above_50": (observed > 50).mean()*100
    })

temporal_diagnostics = pd.DataFrame(temporal_rows)
display(temporal_diagnostics.head())
temporal_diagnostics.to_csv(f"{OUT_DIR}/Exploratory_temporal.csv", index=False)

print("Stations with ACF(1):", temporal_diagnostics.acf_lag1.notna().sum())
print("Stations with ADF result:", temporal_diagnostics.adf_p.notna().sum())
print("Median missing calendar-day percentage:", temporal_diagnostics.missing_calendar_days_pct.median())

### Temporal decision gate

## 14. Past-only feature engineering - one-day-ahead

In [ ]:
                                                                             
predictor_cols = [c for c in [
    "pm25", "AOD", "NO2", "CO", "O3", "temperature_celsius",
    "pressure_mb", "wind_u", "wind_v"
] if c in df.columns]

feature_base = df.sort_values(["id", "date"]).copy()
if feature_base.duplicated(["id", "date"]).any():
    examples = feature_base.loc[
        feature_base.duplicated(["id", "date"], keep=False), ["id", "date"]
    ].head(10)
    raise ValueError(
        "Duplicate station-date records exist. Resolve them using source/domain rules "
        "before daily reindexing. Example duplicates:\\n" + examples.to_string(index=False)
    )
feature_parts = []

for sid, g in feature_base.groupby("id", sort=False):
    g = g.sort_values("date").set_index("date")
                                                                                       
    daily = g.reindex(pd.date_range(g.index.min(), g.index.max(), freq="D"))
    daily["id"] = sid
    daily["sensor_name"] = g["sensor_name"].iloc[0] if "sensor_name" in g.columns and len(g) else pd.NA
    for c in ["latitude", "longitude"]:
        if c in g.columns:
            daily[c] = g[c].median()

                           
    for col in predictor_cols:
        daily[f"{col}_lag1"] = daily[col].shift(1)

                                                                                       
    if "pm25" in daily:
        daily["pm25_roll3_mean"] = daily["pm25"].shift(1).rolling(3, min_periods=3).mean()
        daily["pm25_roll3_std"] = daily["pm25"].shift(1).rolling(3, min_periods=3).std()
        daily["pm25_roll7_mean"] = daily["pm25"].shift(1).rolling(7, min_periods=5).mean()
        daily["pm25_roll7_max"] = daily["pm25"].shift(1).rolling(7, min_periods=5).max()

    for col in [c for c in ["AOD", "temperature_celsius", "NO2"] if c in daily.columns]:
        daily[f"{col}_diff1"] = daily[col].shift(1) - daily[col].shift(2)
        daily[f"{col}_diff3"] = daily[col].shift(1) - daily[col].shift(4)

                                              
    doy = daily.index.dayofyear
    daily["day_of_year_sin"] = np.sin(2*np.pi*doy/365.25)
    daily["day_of_year_cos"] = np.cos(2*np.pi*doy/365.25)
    daily["month"] = daily.index.month
    daily["day_of_week"] = daily.index.dayofweek

    daily["date"] = daily.index
                                                             
    daily = daily[daily["pm25"].notna()].copy()
                                                                                              
    daily = daily[daily["pm25_lag1"].notna()].copy()
    feature_parts.append(daily.reset_index(drop=True))

model_data = pd.concat(feature_parts, ignore_index=True).sort_values(["date", "id"]).reset_index(drop=True)
print("Feature dataset shape:", model_data.shape)
print("Target dates:", model_data.date.min(), "to", model_data.date.max())
print("Stations:", model_data.id.nunique())
display(model_data.head())

model_data.to_csv(f"{OUT_DIR}/PM25_Forecasting_Features_1day.csv", index=False)

## 15. Leakage checks and feature/metadata separation

In [ ]:
assert model_data["pm25"].notna().all()
assert (model_data["date"].notna()).all()
assert model_data["pm25_lag1"].notna().all()

                                                                            
metadata_cols = [c for c in ["date", "id", "sensor_name"] if c in model_data.columns]
target_col = "pm25"
excluded_cols = metadata_cols + [target_col]
feature_cols = [c for c in model_data.columns
                if c not in excluded_cols and pd.api.types.is_numeric_dtype(model_data[c])]

assert "id" not in feature_cols
assert "pm25" not in feature_cols
assert all(not c.endswith("_current") for c in feature_cols)

print("Target:", target_col)
print("Number of numeric candidate features:", len(feature_cols))
print("Candidate features:", feature_cols)
print("Station ID excluded from predictors:", "id" not in feature_cols)
print("Target-day PM2.5 excluded from X:", "pm25" not in feature_cols)

## 16. Preprocessing Pipeline - fold-wise imputation, optional feature selection

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

                                                     
X = model_data[feature_cols].copy()
y = model_data[target_col].copy()
groups = model_data["id"].to_numpy()
dates = pd.to_datetime(model_data["date"])

                                                                          
numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

preprocessor = ColumnTransformer(
    transformers=[("numeric", numeric_preprocess, feature_cols)],
    remainder="drop"
)

                                                                                         
gb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

print(gb_pipeline)
print("Pipeline created but NOT fitted on the full dataset.")

## 17. Outlier-treatment decision and evaluation protocol

In [ ]:
print("Outlier policy:")
print("1) Keep plausible high-PM2.5 events.")
print("2) Correct/remove only documented measurement errors using source/domain evidence.")
print("3) If a statistical threshold is learned, fit it on training fold only.")
print("4) Compare untreated vs chosen treatment using the same validation/test rows.")
print("5) Report MAE, RMSE and R² for both; do not choose a treatment using final-test labels.")

                                                                                 

## 18. Validation split: independent future test period

In [ ]:
TEST_START = pd.Timestamp("2025-04-07")
TEST_END = pd.Timestamp("2026-04-06")

development_data = model_data[model_data.date < TEST_START].copy().reset_index(drop=True)
final_test_data = model_data[model_data.date.between(TEST_START, TEST_END)].copy().reset_index(drop=True)

if development_data.empty or final_test_data.empty:
    raise ValueError("Development or final test set is empty. Check TEST_START/TEST_END against the data coverage.")

assert development_data.date.max() < TEST_START
assert final_test_data.date.min() >= TEST_START
assert final_test_data.date.max() <= TEST_END
assert set(development_data.date).isdisjoint(set(final_test_data.date))

print("Development:", len(development_data), "rows |", development_data.date.min(), "to", development_data.date.max())
print("Final future test:", len(final_test_data), "rows |", final_test_data.date.min(), "to", final_test_data.date.max())
print("Development stations:", development_data.id.nunique())
print("Test stations:", final_test_data.id.nunique())
print("Stations shared:", len(set(development_data.id) & set(final_test_data.id)))
print("Stations only in test:", len(set(final_test_data.id) - set(development_data.id)))

## 19. Temporal CV - 5 folds, one-day gap

In [ ]:
dev = development_data.copy().reset_index(drop=True)
calendar_dates = pd.date_range(dev.date.min(), dev.date.max(), freq="D")
tscv = TimeSeriesSplit(n_splits=5, gap=1)

temporal_folds = {}
temporal_summary_rows = []
for fold, (train_date_idx, valid_date_idx) in enumerate(tscv.split(calendar_dates), start=1):
    train_dates = calendar_dates[train_date_idx]
    valid_dates = calendar_dates[valid_date_idx]
    train_idx = np.flatnonzero(dev.date.isin(train_dates).to_numpy())
    valid_idx = np.flatnonzero(dev.date.isin(valid_dates).to_numpy())

    temporal_folds[f"train_{fold}"] = train_idx
    temporal_folds[f"valid_{fold}"] = valid_idx
    temporal_summary_rows.append({
        "fold": fold, "train_rows": len(train_idx), "validation_rows": len(valid_idx),
        "train_start": dev.iloc[train_idx].date.min(),
        "train_end": dev.iloc[train_idx].date.max(),
        "validation_start": dev.iloc[valid_idx].date.min(),
        "validation_end": dev.iloc[valid_idx].date.max(),
        "calendar_gap_days": (valid_dates.min() - train_dates.max()).days - 1
    })

temporal_summary = pd.DataFrame(temporal_summary_rows)
display(temporal_summary)
with open(f"{OUT_DIR}/folds_temporal.json", "w", encoding="utf-8") as f:
    json.dump({k: v.tolist() for k, v in temporal_folds.items()}, f, indent=2)
assert (temporal_summary["calendar_gap_days"] >= 1).all()

## 20. Spatial CV - GroupKFold by station ID

In [ ]:
gkf = GroupKFold(n_splits=5)
spatial_folds = {}
spatial_summary_rows = []
X_placeholder = np.zeros((len(dev), 1))
dev_groups = dev.id.to_numpy()

for fold, (train_idx, valid_idx) in enumerate(gkf.split(X_placeholder, groups=dev_groups), start=1):
    train_stations = set(dev_groups[train_idx])
    valid_stations = set(dev_groups[valid_idx])
    shared = train_stations & valid_stations
    assert len(shared) == 0
    spatial_folds[f"train_{fold}"] = train_idx
    spatial_folds[f"valid_{fold}"] = valid_idx
    spatial_summary_rows.append({
        "fold": fold, "train_rows": len(train_idx), "validation_rows": len(valid_idx),
        "train_stations": len(train_stations), "validation_stations": len(valid_stations),
        "shared_stations": len(shared)
    })

spatial_summary = pd.DataFrame(spatial_summary_rows)
display(spatial_summary)
with open(f"{OUT_DIR}/folds_spatial.json", "w", encoding="utf-8") as f:
    json.dump({k: v.tolist() for k, v in spatial_folds.items()}, f, indent=2)

## 21. Random K-Fold - diagnostic only (never model selection)

In [ ]:
                                                                                        
                                                                    
random_cv = KFold(n_splits=5, shuffle=True, random_state=42)
random_folds = {}
random_summary_rows = []
for fold, (train_idx, valid_idx) in enumerate(random_cv.split(dev), start=1):
    random_folds[f"train_{fold}"] = train_idx
    random_folds[f"valid_{fold}"] = valid_idx
    random_summary_rows.append({
        "fold": fold, "train_rows": len(train_idx), "validation_rows": len(valid_idx),
        "train_date_min": dev.iloc[train_idx].date.min(),
        "train_date_max": dev.iloc[train_idx].date.max(),
        "validation_date_min": dev.iloc[valid_idx].date.min(),
        "validation_date_max": dev.iloc[valid_idx].date.max()
    })
random_summary = pd.DataFrame(random_summary_rows)
display(random_summary)
with open(f"{OUT_DIR}/folds_standard.json", "w", encoding="utf-8") as f:
    json.dump({k: v.tolist() for k, v in random_folds.items()}, f, indent=2)

## 22. Final validation checks and reusable fold files

In [ ]:
                                                 
checks = []
for fold in range(1, 6):
    tr = dev.iloc[temporal_folds[f"train_{fold}"]]
    va = dev.iloc[temporal_folds[f"valid_{fold}"]]
    checks.append({
        "fold": fold,
        "chronological": tr.date.max() < va.date.min(),
        "calendar_gap_at_least_one_day": (va.date.min() - tr.date.max()).days >= 2,
        "no_date_overlap": set(tr.date).isdisjoint(set(va.date)),
        "final_test_not_in_development": set(va.date).isdisjoint(set(final_test_data.date))
    })
validation_checks = pd.DataFrame(checks)
display(validation_checks)
assert validation_checks.drop(columns="fold").all().all()

                                   
temporal_summary.to_csv(f"{OUT_DIR}/PM25_Temporal_CV_Summary.csv", index=False)
spatial_summary.to_csv(f"{OUT_DIR}/PM25_Spatial_CV_Summary.csv", index=False)
random_summary.to_csv(f"{OUT_DIR}/PM25_Random_KFold_Diagnostic_Summary.csv", index=False)
validation_checks.to_csv(f"{OUT_DIR}/PM25_Validation_Checks.csv", index=False)

print("All fold indices saved. Reuse these exact indices across all models.")